In [ ]:
# ============================================================
# Hotel Reviews Sentiment Analysis
# Notebook 2: Machine Learning
# Dataset: La Veranda Hotel - Booking.com Reviews
# ============================================================

# ============================================================
# SECTION 1: Install Libraries
# ============================================================

!pip install nltk
!pip install textblob
!pip install wordcloud
!pip install seaborn
!pip install joblib
!pip install xgboost

In [ ]:
# ============================================================
# SECTION 2: Import Libraries
# ============================================================

import nltk
import numpy as np
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder

from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                               GradientBoostingClassifier)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)

import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
# ============================================================
# SECTION 3: Load Cleaned Dataset
# ============================================================

df = pd.read_csv('Dataset/cleaned/hotel_reviews_cleaned.csv')
df.head()

In [ ]:
# ============================================================
# SECTION 4: Data Validation
# ============================================================

print("Shape:", df.shape)
df.isnull().sum()

In [ ]:
df.dropna(subset=['cleaned_positive', 'cleaned_negative', 'sentiment'], inplace=True)
df.isnull().sum()

In [ ]:
# ============================================================
# SECTION 5: Exploratory Data Analysis
# ============================================================

# --- 5a. Score Distribution ---
plt.figure(figsize=(10, 6))
sns.histplot(df['Score'], bins=10, kde=True, color='steelblue')
plt.title('Distribution of Guest Scores')
plt.xlabel('Score (out of 10)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# --- 5b. Sentiment Distribution ---
plt.figure(figsize=(8, 5))
sns.countplot(x='sentiment', data=df, palette='Set2',
              order=['negative', 'neutral', 'positive'])
plt.title('Sentiment Distribution (Score-Based Labels)')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.show()

In [ ]:
# --- 5c. Review Length Distribution (Positive vs Negative) ---
df['pos_length'] = df['cleaned_positive'].apply(lambda x: len(str(x).split()))
df['neg_length'] = df['cleaned_negative'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['pos_length'], bins=30, kde=True, ax=axes[0], color='green')
axes[0].set_title('Positive Review Length Distribution')
axes[0].set_xlabel('Number of Words')

sns.histplot(df['neg_length'], bins=30, kde=True, ax=axes[1], color='red')
axes[1].set_title('Negative Review Length Distribution')
axes[1].set_xlabel('Number of Words')
plt.tight_layout()
plt.show()

In [ ]:
# --- 5d. WordCloud - Positive Reviews ---
pos_words = ' '.join(df['cleaned_positive'].dropna().astype(str))
wordcloud_pos = WordCloud(width=800, height=400, background_color='white',
                          colormap='Greens', max_font_size=110,
                          random_state=42).generate(pos_words)
plt.figure(figsize=(15, 7))
plt.imshow(wordcloud_pos, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Positive Reviews', fontsize=16)
plt.show()

In [ ]:
# --- 5e. WordCloud - Negative Reviews ---
neg_words = ' '.join(df['cleaned_negative'].dropna().astype(str))
wordcloud_neg = WordCloud(width=800, height=400, background_color='white',
                          colormap='Reds', max_font_size=110,
                          random_state=42).generate(neg_words)
plt.figure(figsize=(15, 7))
plt.imshow(wordcloud_neg, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Negative Reviews', fontsize=16)
plt.show()

In [ ]:
# --- 5f. Polarity Distribution (Positive vs Negative track) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['polarity_positive'], bins=20, color='green', edgecolor='black')
axes[0].set_title('Polarity Distribution - Positive Reviews')
axes[0].set_xlabel('Polarity')

axes[1].hist(df['polarity_negative'], bins=20, color='red', edgecolor='black')
axes[1].set_title('Polarity Distribution - Negative Reviews')
axes[1].set_xlabel('Polarity')
plt.tight_layout()
plt.show()

In [ ]:
# --- 5g. Top 10 Guest Countries ---
plt.figure(figsize=(12, 5))
df['GuestCountry'].value_counts().head(10).plot(kind='bar', color='steelblue')
plt.title('Top 10 Guest Countries')
plt.xlabel('Country')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5h. Sentiment by Group Type ---
plt.figure(figsize=(10, 6))
sns.countplot(x='GroupType', hue='sentiment', data=df, palette='Set2')
plt.title('Sentiment by Group Type')
plt.xlabel('Group Type')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 6: Vectorization & Train/Test Split
# We run two parallel pipelines:
#   Track A — Positive Reviews
#   Track B — Negative Reviews
# ============================================================

# --- Track A: Positive Reviews ---
vectorizer_pos = TfidfVectorizer()
X_pos = vectorizer_pos.fit_transform(df['cleaned_positive'].astype(str))
y = df['sentiment']

X_train_pos, X_test_pos, y_train, y_test = train_test_split(
    X_pos, y, test_size=0.2, random_state=42, stratify=y)

print("Positive Track — Train:", X_train_pos.shape, "| Test:", X_test_pos.shape)

# --- Track B: Negative Reviews ---
vectorizer_neg = TfidfVectorizer()
X_neg = vectorizer_neg.fit_transform(df['cleaned_negative'].astype(str))

X_train_neg, X_test_neg, _, _ = train_test_split(
    X_neg, y, test_size=0.2, random_state=42, stratify=y)

print("Negative Track — Train:", X_train_neg.shape, "| Test:", X_test_neg.shape)

In [ ]:
# ============================================================
# SECTION 7: Helper Function
# Keeps model training/evaluation clean and DRY
# ============================================================

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    print(f"Evaluation for: {name}".center(65, '_'))
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted')
    rec  = recall_score(y_test, preds, average='weighted')
    f1   = f1_score(y_test, preds, average='weighted')

    print(f"Accuracy:  {acc:.2%}")
    print(f"Precision: {prec:.2%}")
    print(f"Recall:    {rec:.2%}")
    print(f"F1-Score:  {f1:.2%}")

    # Confusion Matrix
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='.0f', linewidths=0.7,
                square=True, cmap='Blues_r',
                xticklabels=['negative','neutral','positive'],
                yticklabels=['negative','neutral','positive'])
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.title(f'{name} — Confusion Matrix')
    plt.tight_layout()
    plt.show()

    print(f"Classification Report — {name}:")
    print('-' * 65)
    print(classification_report(y_test, preds))

    return model, preds, [acc*100, prec*100, rec*100, f1*100]

In [ ]:
!pip install threadpoolctl --upgrade

In [ ]:
# ============================================================
# SECTION 8: Train All Models
# Each model is trained on BOTH tracks separately
# ============================================================

models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000),
    'AdaBoost':               AdaBoostClassifier(),
    'Decision Tree':          DecisionTreeClassifier(),
    'Random Forest':          RandomForestClassifier(),
    'Gradient Boosting':      GradientBoostingClassifier(),
    'C-Support Vector':       SVC(),
}

scores_pos = {}
scores_neg = {}
trained_pos = {}
trained_neg = {}

for name, model_blueprint in models.items():
    import copy
    print("\n" + "="*65)
    print(f"  POSITIVE REVIEW TRACK — {name}")
    print("="*65)
    m_pos, _, s_pos = evaluate_model(
        name, copy.deepcopy(model_blueprint),
        X_train_pos, X_test_pos, y_train, y_test)
    trained_pos[name] = m_pos
    scores_pos[name] = s_pos

    print("\n" + "="*65)
    print(f"  NEGATIVE REVIEW TRACK — {name}")
    print("="*65)
    m_neg, _, s_neg = evaluate_model(
        name, copy.deepcopy(model_blueprint),
        X_train_neg, X_test_neg, y_train, y_test)
    trained_neg[name] = m_neg
    scores_neg[name] = s_neg

In [ ]:
# ============================================================
# SECTION 9: Model Comparison Chart
# ============================================================

def plot_model_comparison(scores_dict, track_name):
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    model_names = list(scores_dict.keys())
    x_pos = np.arange(len(metrics))
    bar_width = 0.10

    def add_labels(bars):
        for bar in bars:
            h = bar.get_height()
            plt.text(bar.get_x() + bar.get_width() / 2, h,
                     f'{h:.1f}%', ha='center', va='bottom',
                     rotation=90, fontsize=7)

    plt.figure(figsize=(20, 10), dpi=150, facecolor='w')
    plt.title(f"Model Evaluation Scores — {track_name}", fontsize=16)

    for i, name in enumerate(model_names):
        bars = plt.bar(x_pos + bar_width * i, scores_dict[name],
                       width=bar_width, label=name)
        add_labels(bars)

    plt.xticks(x_pos + bar_width * (len(model_names)/2), metrics, fontsize=13)
    plt.yticks(range(0, 110, 10), fontsize=11)
    plt.xlabel('Evaluation Metrics', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.legend(bbox_to_anchor=(1.01, 1), borderaxespad=0.)
    plt.tight_layout()
    plt.show()

plot_model_comparison(scores_pos, 'Positive Review Track')
plot_model_comparison(scores_neg, 'Negative Review Track')

In [ ]:
# ============================================================
# SECTION 10: Save Best Models & Vectorizers
# ============================================================

import os
os.makedirs('Models', exist_ok=True)

# Save all models for both tracks
for name, model in trained_pos.items():
    fname = name.lower().replace(' ', '_').replace('-', '')
    joblib.dump(model, f'Models/{fname}_positive.pkl')

for name, model in trained_neg.items():
    fname = name.lower().replace(' ', '_').replace('-', '')
    joblib.dump(model, f'Models/{fname}_negative.pkl')

# Save vectorizers
joblib.dump(vectorizer_pos, 'Models/tfidf_vectorizer_positive.pkl')
joblib.dump(vectorizer_neg, 'Models/tfidf_vectorizer_negative.pkl')

print("All models and vectorizers saved to Models/")